In [5]:
import os
import gc
import cv2
import math
import json
import torch
import paddle
from pathlib import Path
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

if not hasattr(Image, "ANTIALIAS"):
    Image.ANTIALIAS = Image.Resampling.LANCZOS
# Import PaddleOCR
from paddleocr import PaddleOCR

# Import VietOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

import warnings
warnings.filterwarnings("ignore")

In [6]:
ROOT = Path(".")
FRAME_PATH = Path("..") / "data" / "frame"
OUTPUT_PATH = ROOT / "OCR.json"
VISUAL_PATH = ROOT / "visualize"

### Tải ảnh và thực hiện OCR

In [7]:
def get_rotated_crop_image(img, points):
    """
    Cắt ảnh dựa trên 4 điểm tọa độ từ PaddleOCR. Xử lý cả box bị xoay/nghiêng.
    """
    points = np.array(points, dtype=np.float32)
    rect = cv2.minAreaRect(points)
    box = cv2.boxPoints(rect)
    box = box.astype(np.int32)

    width = int(rect[1][0])
    height = int(rect[1][1])

    src_pts = points.astype("float32")
    dst_pts = np.array([[0, 0],
                        [width - 1, 0],
                        [width - 1, height - 1],
                        [0, height - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(img, M, (width, height))
    
    if height > width * 1.2:
        warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)
        
    return warped

def init_models(use_gpu=False):
    """
    Khởi tạo và trả về các mô hình OCR.
    Gọi hàm này MỘT LẦN duy nhất khi bắt đầu chương trình để tối ưu hiệu suất.
    """
    print("Đang khởi tạo các mô hình Deep Learning...")
    
    # 1. PaddleOCR Detection Model
    det_model = PaddleOCR(use_angle_cls=False, lang='vi')
    
    # 2. VietOCR Recognition Model
    config = Cfg.load_config_from_name('vgg_seq2seq')
    config['device'] = 'cuda:0' if (use_gpu and torch.cuda.is_available()) else 'cpu'
    recognizer = Predictor(config)
    
    print("Khởi tạo mô hình hoàn tất!\n" + "="*40)
    return det_model, recognizer

def process_single_image(image_path, det_model, recognizer, save_visual_path=None):
    """
    Hàm chức năng: Thực hiện pipeline OCR trên một ảnh cụ thể.
    Trả về một chuỗi văn bản hoàn chỉnh.
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"  [Lỗi] Không thể đọc ảnh: {image_path}")
        return ""
        
    img_draw = img.copy() if save_visual_path else None
    
    # Bước 1: Detection
    result = det_model.ocr(img)
    if not result:
        return ""
    first_result = result[0]
    if isinstance(first_result, dict):
        boxes = first_result.get('dt_polys', [])
    else:
        boxes = first_result
    
    if not boxes:
        return "" # Không tìm thấy chữ
        
    full_text_list = []
    
    # Bước 2 & 3: Crop và Recognition
    for points in boxes:
        # Cắt ảnh
        cropped_img_cv = get_rotated_crop_image(img, points)
        cropped_img_rgb = cv2.cvtColor(cropped_img_cv, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_img_rgb)
        
        # Nhận dạng tiếng Việt
        try:
            text = recognizer.predict(pil_img)
        except Exception as e:
            print(f"  [Cảnh báo] Lỗi nội bộ VietOCR khi đọc patch: {e}")
            continue
            
        if text.strip():
            full_text_list.append(text)
            
        # Tùy chọn: Vẽ khung nếu cần lưu kết quả trực quan
        if save_visual_path:
            pts = np.array(points, np.int32).reshape((-1, 1, 2))
            cv2.polylines(img_draw, [pts], isClosed=True, color=(0, 0, 255), thickness=2)

    if save_visual_path and img_draw is not None:
        # Đảm bảo thư mục lưu tồn tại
        os.makedirs(os.path.dirname(save_visual_path), exist_ok=True)
        cv2.imwrite(save_visual_path, img_draw)

    return " ".join(full_text_list)


def process_folder(frame_path, output_file, visualize_dir=None, use_gpu=True):
    folders = os.listdir(frame_path)
    if not folders:
        print(f"Không tìm thấy thư mục ảnh nào trong: {folders}")
    else:
        print(f"Tìm thấy {len(folders)} thư mục ảnh.")

    ocr_results = {}

    det_model, recognizer = init_models(use_gpu)

    total_processed_images = 0

    for folder in tqdm(folders, "Đang xử lý các folder"):
        print(f"\nĐang xử lý thư mục: {folder}")
        folder_path = os.path.join(frame_path, folder)

        image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.webp')])
        for image_file_name in tqdm(image_files, "Đang xử lý các ảnh"):
            image_path = os.path.join(folder_path, image_file_name)

            # print(f"Đang sử lý ảnh tại đường dẫn: {image_path}")

            text_result = process_single_image(
                image_path,
                det_model,
                recognizer,
                save_visual_path=os.path.join(visualize_dir, folder, image_file_name) if visualize_dir else None
            )

            frame_idx = image_file_name.removeprefix('keyframe_').removesuffix('.webp')
            key = f"{folder}_{frame_idx}"
            ocr_results[key] = text_result

            total_processed_images += 1

            if total_processed_images % 100 == 0:
              print(f"\n--- Đã xử lý {total_processed_images} ảnh. Đang lưu tạm JSON và dọn dẹp RAM/VRAM... ---")
            
              # 1. Lưu tạm JSON (Ghi đè file cũ)
              with open(output_file, 'w', encoding='utf-8') as f_out:
                  json.dump(ocr_results, f_out, ensure_ascii=False, indent=4)
                  
              # 2. Dọn rác RAM hệ thống
              gc.collect() 
              
              # 3. Dọn rác VRAM của GPU (Rất quan trọng cho PyTorch)
              if use_gpu and torch.cuda.is_available():
                  torch.cuda.empty_cache() 
            # ---------------------------------------------------

    # LƯU FILE JSON LẦN CUỐI CÙNG SAU KHI QUÉT XONG TẤT CẢ
    print(f"\n[Đang hoàn tất] Đang lưu file JSON cuối cùng...")
    with open(output_file, 'w', encoding='utf-8') as f_out:
        json.dump(ocr_results, f_out, ensure_ascii=False, indent=4)
        
    print(f"\n[HOÀN TẤT] Đã xử lý tổng cộng {total_processed_images} ảnh. Kết quả cuối cùng lưu tại: {output_file}")


In [8]:
process_folder(FRAME_PATH, OUTPUT_PATH, visualize_dir=VISUAL_PATH)

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/pintee/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/pintee/.paddlex/official_models/UVDoc`.


Tìm thấy 1 thư mục ảnh.
Đang khởi tạo các mô hình Deep Learning...


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/pintee/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('latin_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/pintee/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


Model weight /tmp/vgg_seq2seq.pth exsits. Ignore download!
Khởi tạo mô hình hoàn tất!


Đang xử lý các folder:   0%|          | 0/1 [00:00<?, ?it/s]


Đang xử lý thư mục: L02_V016


Đang xử lý các folder: 100%|██████████| 1/1 [00:02<00:00,  2.74s/it]


[Đang hoàn tất] Đang lưu file JSON cuối cùng...

[HOÀN TẤT] Đã xử lý tổng cộng 9 ảnh. Kết quả cuối cùng lưu tại: OCR.json
